In [21]:
import sys

!{sys.executable} -m pip install graphdatascience pandas


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip3 install --upgrade pip


En esta parte de la práctica se trabaja con el cliente Python de Graph Data Science para analizar las armas como un grafo de similitud. La idea es que dos armas sean parecidas si necesitan materiales comunes para fabricarse o actualizarse. A partir de esa idea, se crea una relación SIMILAR entre armas, se agrupan las armas en comunidades usando Leiden y, finalmente, se aplica PageRank dentro de cada comunidad para elegir sus representantes.

Primero se importan las librerías necesarias: GraphDataScience para comunicarnos con Neo4j/GDS, pandas para manejar tablas y display/Markdown para mostrar mejor los resultados en el notebook.

La conexión se hace con:

In [22]:
from graphdatascience import GraphDataScience
import pandas as pd
from IPython.display import display, Markdown

gds = GraphDataScience(
    "bolt://localhost:7687",
    auth=("neo4j", "password")
)

print("Conectado a Neo4j")
print("Versión GDS:", gds.version())

Conectado a Neo4j
Versión GDS: 2.12.0


Esto conecta el notebook con la base de datos Neo4j. En el resultado se comprueba que la conexión funciona correctamente, ya que aparece el mensaje "Conectado a Neo4j" y la versión de GDS instalada, que en nuestro caso es la 2.12.0. Esta comprobación es importante porque toda esta parte depende de que el plugin de Graph Data Science esté disponible.

Después se define la función ejecutar_cypher. Esta función simplemente nos permite ejecutar consultas Cypher desde Python y mostrar el resultado en forma de tabla. La usamos para no repetir siempre el mismo código cada vez que queremos lanzar una consulta.

In [23]:
def ejecutar_cypher(titulo, query, params=None):
    display(Markdown(f"## {titulo}"))
    df = gds.run_cypher(query, params=params or {})
    
    if df.empty:
        print("La consulta no devolvió resultados.")
    else:
        display(df)
    
    return df

Antes de crear las nuevas relaciones SIMILAR, se eliminan las que pudieran existir de ejecuciones anteriores:

In [24]:
query_limpiar_similar = """
MATCH ()-[s:SIMILAR]-()
WITH collect(DISTINCT s) AS relaciones
FOREACH (r IN relaciones | DELETE r)
RETURN size(relaciones) AS relacionesEliminadas
"""

df_limpieza = ejecutar_cypher(
    "Limpiar relaciones SIMILAR anteriores",
    query_limpiar_similar
)

## Limpiar relaciones SIMILAR anteriores

,relacionesEliminadas
0,28140


Aquí usamos collect(DISTINCT s) para guardar cada relación SIMILAR una sola vez. Esto evita problemas con relaciones repetidas, ya que al buscar sin dirección una relación podría aparecer más de una vez. Después, FOREACH recorre la lista de relaciones y las borra.

### Paso 1
- El primer paso es escribir una consulta que cree una arista nueva entre las armas. 
- Dicha arista se llamará SIMILAR y unirá dos armas si tienen al menos un Item en común entre sus
procesos de creación o actualización. La arista tendrá un único atributo llamado weight
que contendrá un valor entre 0 y 1. Dicho valor será el índice de Jaccard entre los objetos
que se necesitan para construir ambas armas.
- Por ejemplo, si un arma requiere estos tres items ["Dragonita", "Cola de Quematrice",
"Piedra de fuego"] y otra arma requiere ["Cristal de tierra", "Machalita", "Dragonita"], la
arista que una ambas armas tendrá un peso de (1/5 = 0.2).

In [25]:
query_crear_similar = """
MATCH (w:Weapon)

CALL {
    WITH w

    OPTIONAL MATCH (wc:WeaponCraft)-[:PRODUCES]->(w)
    OPTIONAL MATCH (wc)-[:NEEDS]->(itemCraft:Item)

    WITH w,
         collect(DISTINCT itemCraft) AS itemsCraft

    OPTIONAL MATCH (wu:WeaponUpgrade)-[:TRANSFORMS]->(w)
    OPTIONAL MATCH (wu)-[:NEEDS]->(itemUpgrade:Item)

    WITH itemsCraft,
         collect(DISTINCT itemUpgrade) AS itemsUpgrade

    WITH itemsCraft + itemsUpgrade AS rawItems

    WITH reduce(
        acc = [],
        item IN rawItems |
        CASE
            WHEN item IS NULL OR item IN acc THEN acc
            ELSE acc + item
        END
    ) AS items

    RETURN items
}

WITH collect({weapon: w, items: items}) AS weaponItems

UNWIND weaponItems AS a
UNWIND weaponItems AS b

WITH a.weapon AS w1,
     a.items AS items1,
     b.weapon AS w2,
     b.items AS items2

WHERE id(w1) < id(w2)
  AND size(items1) > 0
  AND size(items2) > 0

WITH w1,
     w2,
     items1,
     items2,
     [i IN items1 WHERE i IN items2] AS interseccion,
     items1 + [i IN items2 WHERE NOT i IN items1] AS unionItems

WHERE size(interseccion) > 0

WITH w1,
     w2,
     toFloat(size(interseccion)) / size(unionItems) AS jaccard

MERGE (w1)-[s:SIMILAR]->(w2)
SET s.weight = jaccard

RETURN count(s) AS relacionesCreadas,
       min(jaccard) AS pesoMinimo,
       max(jaccard) AS pesoMaximo,
       avg(jaccard) AS pesoMedio
"""

df_similar = ejecutar_cypher(
    "Paso 1. Crear relaciones SIMILAR con peso Jaccard",
    query_crear_similar
)

## Paso 1. Crear relaciones SIMILAR con peso Jaccard

,relacionesCreadas,pesoMinimo,pesoMaximo,pesoMedio
0,28140,0.125,1.0,0.322835


El primer paso crea relaciones `SIMILAR` entre armas que comparten al menos un material, ya sea de creación (`WeaponCraft`) o de actualización (`WeaponUpgrade`). Para cada arma se recopilan sus materiales, se eliminan nulos y repetidos, y después se comparan las armas por pares.

Para evitar duplicados, se usa:

```cypher
WHERE id(w1) < id(w2)
```

Así no se compara `A-B` y luego `B-A`, ni un arma consigo misma.

La similitud se calcula con Jaccard:

```cypher
items comunes / items totales distintos
```

Por eso se calcula la intersección y la unión de materiales. Solo se crea `SIMILAR` si la intersección no está vacía, es decir, si las armas comparten algún material.

El peso se guarda en `weight`:

```cypher
MERGE (w1)-[s:SIMILAR]->(w2)
SET s.weight = jaccard
```

Aunque la relación se guarde con una dirección, conceptualmente es simétrica: si un arma es similar a otra, la similitud vale en ambos sentidos.

El resultado fue coherente: se crearon `28140` relaciones, con pesos entre `0.125` y `1.0`. Además, las comprobaciones dieron `0` pesos inválidos y `0` relaciones de un arma consigo misma, por lo que el cálculo parece correcto:


In [26]:
check_2 = """
MATCH (:Weapon)-[s:SIMILAR]->(:Weapon)
WHERE s.weight IS NULL
   OR s.weight <= 0
   OR s.weight > 1
RETURN count(s) AS relacionesInvalidas
"""

df_check_2 = ejecutar_cypher(
    "Check 2. Pesos inválidos",
    check_2
)

## Check 2. Pesos inválidos

,relacionesInvalidas
0,0


In [27]:
check_3 = """
MATCH (w:Weapon)-[s:SIMILAR]->(w)
RETURN count(s) AS relacionesConsigoMisma
"""

df_check_3 = ejecutar_cypher(
    "Check 3. Relaciones de un arma consigo misma",
    check_3
)

## Check 3. Relaciones de un arma consigo misma

,relacionesConsigoMisma
0,0


In [28]:
check_4 = """
MATCH (w1:Weapon)-[s:SIMILAR]->(w2:Weapon)
RETURN w1.name AS arma1,
       w2.name AS arma2,
       s.weight AS jaccard
ORDER BY jaccard DESC
LIMIT 20
"""

df_check_4 = ejecutar_cypher(
    "Check 4. Top relaciones SIMILAR por peso",
    check_4
)

## Check 4. Top relaciones SIMILAR por peso

,arma1,arma2,jaccard
0,Cañón de esperanza V,Segur de esperanza V,1.0
1,Arco de cazador I,Aspa ósea I,1.0
2,Arco de esperanza IV,Segur de esperanza IV,1.0
3,Punzadora potente,Arco poder salvaje I,1.0
4,Arco de esperanza V,Segur de esperanza V,1.0
5,Cañón de esperanza III,Segur de esperanza III,1.0
6,Arco de esperanza III,Segur de esperanza III,1.0
7,Cañón de esperanza IV,Segur de esperanza IV,1.0
8,Uth Dalgap II,Uth Khviluk II,1.0
9,Uth Dalgap III,Uth Khviluk III,1.0


Esta comprobación permite revisar las relaciones `SIMILAR` con mayor peso. Al ordenar por `jaccard` de forma descendente, aparecen primero las armas más parecidas. En este caso, las primeras relaciones tienen peso `1.0`, lo que significa que esas parejas de armas necesitan exactamente los mismos materiales. Esto confirma que el cálculo de similitud está funcionando correctamente, ya que el valor máximo de Jaccard representa una coincidencia total entre los conjuntos de materiales.

In [29]:
def borrar_grafo_si_existe(nombre_grafo):
    df = gds.run_cypher(
        """
        CALL gds.graph.exists($graphName)
        YIELD exists
        RETURN exists
        """,
        params={"graphName": nombre_grafo}
    )
    
    existe = bool(df["exists"].iloc[0])
    
    if existe:
        gds.run_cypher(
            """
            CALL gds.graph.drop($graphName)
            YIELD graphName
            RETURN graphName
            """,
            params={"graphName": nombre_grafo}
        )
        print(f"Grafo eliminado: {nombre_grafo}")

Esta función sirve para **borrar un grafo GDS proyectado en memoria si ya existe**.

Primero comprueba si existe un grafo con ese nombre:

```cypher
CALL gds.graph.exists($graphName)
```

Si existe, lo borra con:

```cypher
CALL gds.graph.drop($graphName)
```

Esto es útil porque, si ejecutamos varias veces el notebook, Neo4j puede dar error al intentar crear un grafo GDS con un nombre que ya existe.

**No borra nodos ni relaciones reales de la base de datos**, solo borra la proyección temporal que usa Graph Data Science en memoria.


### Paso 2

* El segundo paso es calcular las comunidades usando el algoritmo de leiden y escribir en cada nodo un atributo que indique a que comunidad pertenece.

In [30]:
nombre_grafo = "weapons_similarity"

borrar_grafo_si_existe(nombre_grafo)

G, resultado_proyeccion = gds.graph.project(
    nombre_grafo,
    "Weapon",
    {
        "SIMILAR": {
            "orientation": "UNDIRECTED",
            "properties": "weight"
        }
    }
)

print("Grafo proyectado correctamente")
print("Nodos:", G.node_count())
print("Relaciones:", G.relationship_count())

display(pd.DataFrame([resultado_proyeccion]))

Grafo proyectado correctamente
Nodos: 1024
Relaciones: 56280


,nodeProjection,relationshipProjection,graphName,nodeCount,relationshipCount,projectMillis
0,"{'Weapon': {'label': 'Weapon', 'properties': {}}}","{'SIMILAR': {'aggregation': 'DEFAULT', 'orient...",weapons_similarity,1024,56280,26


In [31]:
nombre_grafo = "weapons_similarity"

borrar_grafo_si_existe(nombre_grafo)

G, resultado_proyeccion = gds.graph.project(
    nombre_grafo,
    "Weapon",
    {
        "SIMILAR": {
            "orientation": "UNDIRECTED",
            "properties": "weight"
        }
    }
)

print("Grafo proyectado correctamente")
print("Nodos:", G.node_count())
print("Relaciones:", G.relationship_count())

display(pd.DataFrame([resultado_proyeccion]))

Grafo eliminado: weapons_similarity
Grafo proyectado correctamente
Nodos: 1024
Relaciones: 56280


,nodeProjection,relationshipProjection,graphName,nodeCount,relationshipCount,projectMillis
0,"{'Weapon': {'label': 'Weapon', 'properties': {}}}","{'SIMILAR': {'aggregation': 'DEFAULT', 'orient...",weapons_similarity,1024,56280,14


In [32]:
resultado_leiden = gds.leiden.write(
    G,
    writeProperty="comunidad",
    relationshipWeightProperty="weight",
    randomSeed=19
)

display(Markdown("## Paso 2. Resultado Leiden"))
display(pd.DataFrame([resultado_leiden]))

## Paso 2. Resultado Leiden

,writeMillis,nodePropertiesWritten,ranLevels,didConverge,nodeCount,communityCount,communityDistribution,modularity,modularities,postProcessingMillis,preProcessingMillis,computeMillis,configuration
0,9,1024,3,True,1024,58,"{'min': 1, 'p5': 1, 'max': 128, 'p999': 128, '...",0.790631,"[0.5669640583391325, 0.7881316664679029, 0.790...",2,1,161,"{'writeProperty': 'comunidad', 'randomSeed': 1..."


In [33]:
query_comunidades = """
MATCH (w:Weapon)
WHERE w.comunidad IS NOT NULL
RETURN w.comunidad AS comunidad,
       count(w) AS numArmas,
       collect(w.name)[0..10] AS ejemplos
ORDER BY comunidad
"""

df_comunidades = ejecutar_cypher(
    "Comunidades detectadas",
    query_comunidades
)

## Comunidades detectadas

,comunidad,numArmas,ejemplos
0,0,14,"[Arco de cazador I, Aspa ósea I, Hachas óseas ..."
1,1,34,"[Puño de Scylla I, Arco de cazador II, Aspa ós..."
2,2,35,"[Cañón de esperanza III, Arco de esperanza III..."
3,3,21,"[Horadadragones II, Hiperguarda II, Hachas dob..."
4,4,42,"[Cañón de esperanza V, Arco de esperanza V, Ho..."
5,5,14,"[Lluvia bendita I, Taladro de Mizuniya I, Aman..."
6,6,1,[Lanza de esperanza I]
7,7,14,"[Cañón de esperanza II, Arco de esperanza II, ..."
8,8,11,"[Escudo Barina III, Falces Barina III, Filo pl..."
9,9,8,"[Terror Dosha I, Azotarraudos Dosha I, Azotede..."


Para poder aplicar algoritmos GDS, primero hay que proyectar el grafo en memoria. Se proyectan los nodos Weapon y las relaciones SIMILAR:

G, resultado_proyeccion = gds.graph.project(
    nombre_grafo,
    "Weapon",
    {
        "SIMILAR": {
            "orientation": "UNDIRECTED",
            "properties": "weight"
        }
    }
)

Aquí es importante que la relación SIMILAR se proyecte como UNDIRECTED. Aunque en Neo4j la relación se haya guardado como w1 -> w2, la similitud no tiene dirección real. Por eso le decimos a GDS que la trate como una conexión no dirigida.

El grafo proyectado tiene:

- 1024 nodos
- 56280 relaciones

Hay 56280 relaciones en la proyección porque al tratar SIMILAR como no dirigida, GDS considera las conexiones en ambos sentidos internamente. Como antes se habían creado 28140 relaciones SIMILAR, la proyección no dirigida representa el doble de direcciones.

Después se aplica Leiden:

resultado_leiden = gds.leiden.write(
    G,
    writeProperty="comunidad",
    relationshipWeightProperty="weight",
    randomSeed=19
)

Leiden es un algoritmo de detección de comunidades. En este caso agrupa armas que están muy conectadas entre sí por relaciones SIMILAR. La propiedad weight hace que las relaciones con mayor Jaccard tengan más importancia en la formación de comunidades.

Se usa writeProperty="comunidad" para escribir en cada nodo Weapon el identificador de la comunidad a la que pertenece. Es decir, después de ejecutar Leiden, cada arma tendrá una propiedad comunidad.

El resultado fue:

- nodePropertiesWritten: 1024
- communityCount: 58
- didConverge: True
- ranLevels: 3
- modularity: 0.790631

Esto significa que Leiden asignó comunidad a las 1024 armas y detectó 58 comunidades. El valor de modularity es bastante alto, lo que indica que las comunidades están razonablemente bien separadas: las armas dentro de una misma comunidad comparten más materiales entre sí que con armas de otras comunidades.

Después se consultan las comunidades detectadas. Se devuelve para cada comunidad el número de armas y algunos ejemplos. Se usa collect(w.name)[0..10] para mostrar solo hasta 10 nombres de ejemplo y no llenar la tabla con listas demasiado largas.


### Paso 3

* El tercer paso es aplicar el algoritmo de pageRank a cada comunidad para obtener la o las
armas con más centralidad y escribir por pantalla sus nombres. Este conjunto de armas
será el representante de cada comunidad.

En esta parte aplicamos PageRank dentro de cada comunidad detectada por Leiden. La idea es escoger, dentro de cada grupo de armas similares, cuál o cuáles son las armas más centrales. Estas armas se consideran representantes de su comunidad porque están bien conectadas con otras armas parecidas mediante relaciones `SIMILAR`.

Como `SIMILAR` representa una similitud entre armas, no tiene una dirección real. Si un arma A es similar a un arma B, también B es similar a A. Por eso usamos una proyección `UNDIRECTED`, para que GDS trate la relación como bidireccional.

In [34]:
# Proyectamos de nuevo el grafo, ahora incluyendo la propiedad comunidad
# para poder filtrar subgrafos por comunidad.

nombre_grafo_pagerank = "weapons_similarity_pagerank"

borrar_grafo_si_existe(nombre_grafo_pagerank)

G_pr, resultado_proyeccion_pr = gds.graph.project(
    nombre_grafo_pagerank,
    {
        "Weapon": {
            "properties": ["comunidad"]
        }
    },
    {
        "SIMILAR": {
            "orientation": "UNDIRECTED",
            "properties": "weight"
        }
    }
)

print("Grafo para PageRank proyectado correctamente")
print("Nodos:", G_pr.node_count())
print("Relaciones:", G_pr.relationship_count())

display(pd.DataFrame([resultado_proyeccion_pr]))

Grafo para PageRank proyectado correctamente
Nodos: 1024
Relaciones: 56280


,nodeProjection,relationshipProjection,graphName,nodeCount,relationshipCount,projectMillis
0,"{'Weapon': {'label': 'Weapon', 'properties': {...","{'SIMILAR': {'aggregation': 'DEFAULT', 'orient...",weapons_similarity_pagerank,1024,56280,20


#### Proyección del grafo para PageRank

Para aplicar PageRank, primero se proyecta un grafo GDS nuevo llamado `weapons_similarity_pagerank`. En esta proyección usamos los nodos `Weapon` y las relaciones `SIMILAR`.

Además, incluimos la propiedad `comunidad` en los nodos, porque después la necesitaremos para filtrar cada comunidad por separado. La relación `SIMILAR` se proyecta con `orientation: "UNDIRECTED"`, ya que la similitud entre armas no tiene dirección.

El resultado de la proyección fue:

- Nodos: `1024`
- Relaciones: `56280`

Aunque en Neo4j se habían creado `28140` relaciones `SIMILAR`, en la proyección aparecen `56280` porque al tratar el grafo como no dirigido, GDS considera internamente las conexiones en ambos sentidos.

### Creación de subgrafos por comunidad

Después de proyectar el grafo global, se recorre cada comunidad detectada por Leiden. Para cada comunidad se crea un subgrafo usando `gds.graph.filter`.

El filtro usado es:

```python
nodeFilter = f"n.comunidad = {comunidad}"

Esto significa que el subgrafo solo contiene las armas que pertenecen a esa comunidad concreta. El parámetro '*' indica que se mantienen todas las relaciones entre los nodos que pasan el filtro.

Este método es más limpio que duplicar relaciones manualmente con UNION ALL, porque ya hemos proyectado SIMILAR como no dirigida desde el principio.

#### Aplicación de PageRank

Una vez creado el subgrafo de una comunidad, se aplica PageRank sobre él:

```python
df_pagerank = gds.pageRank.stream(
    Gsub,
    relationshipWeightProperty="weight"
)

El algoritmo usa la propiedad weight de SIMILAR, que representa el índice de Jaccard entre los materiales de dos armas. Así, las conexiones entre armas más parecidas tienen más importancia.

Después se busca el valor máximo de PageRank dentro de la comunidad. Si varias armas tienen el mismo valor máximo, se devuelven todas como representantes. Esto permite tener en cuenta empates.

In [35]:
comunidades = gds.run_cypher("""
MATCH (w:Weapon)
WHERE w.comunidad IS NOT NULL
RETURN DISTINCT w.comunidad AS comunidad
ORDER BY comunidad
""")

representantes = []

for comunidad in comunidades["comunidad"]:
    nombre_subgrafo = f"weapons_comunidad_{comunidad}"
    borrar_grafo_si_existe(nombre_subgrafo)

    # Creamos un subgrafo filtrando solo los nodos de esa comunidad.
    # Como el grafo base ya es UNDIRECTED, no hace falta duplicar relaciones con UNION ALL.
    query_filtrar_subgrafo = """
    CALL gds.graph.filter(
        $subgraphName,
        $graphName,
        $nodeFilter,
        '*'
    )
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount
    """

    info_subgrafo = gds.run_cypher(
        query_filtrar_subgrafo,
        params={
            "subgraphName": nombre_subgrafo,
            "graphName": nombre_grafo_pagerank,
            "nodeFilter": f"n.comunidad = {comunidad}"
        }
    )

    display(Markdown(f"### Comunidad {comunidad}"))
    display(info_subgrafo)

    Gsub = gds.graph.get(nombre_subgrafo)

    df_pagerank = gds.pageRank.stream(
        Gsub,
        relationshipWeightProperty="weight"
    )

    if not df_pagerank.empty:
        max_score = df_pagerank["score"].max()

        top_nodes = df_pagerank[
            df_pagerank["score"] == max_score
        ]["nodeId"].astype(int).tolist()

        nombres = gds.run_cypher(
            """
            UNWIND $ids AS nodeId
            MATCH (w:Weapon)
            WHERE id(w) = nodeId
            RETURN w.name AS arma
            ORDER BY arma
            """,
            params={"ids": top_nodes}
        )

        representantes.append({
            "comunidad": comunidad,
            "armas_representantes": nombres["arma"].tolist(),
            "pagerank": max_score,
            "num_nodos": int(info_subgrafo["nodeCount"].iloc[0]),
            "num_relaciones": int(info_subgrafo["relationshipCount"].iloc[0])
        })

    gds.graph.drop(Gsub)

### Comunidad 0

,graphName,nodeCount,relationshipCount
0,weapons_comunidad_0,14,182


### Comunidad 1

,graphName,nodeCount,relationshipCount
0,weapons_comunidad_1,34,590


### Comunidad 2

...

,graphName,nodeCount,relationshipCount
0,weapons_comunidad_60,7,42


### Comunidad 62

,graphName,nodeCount,relationshipCount
0,weapons_comunidad_62,14,182


### Comunidad 63

,graphName,nodeCount,relationshipCount
0,weapons_comunidad_63,14,182


Estos resultados muestran los **subgrafos creados para cada comunidad** antes de aplicar PageRank.

`nodeCount` indica cuántas armas hay en la comunidad y `relationshipCount` cuántas relaciones `SIMILAR` hay entre ellas.

Se ve que hay comunidades grandes y muy conectadas, como la comunidad 35, y otras comunidades con una sola arma y `0` relaciones. Esto no es un error: simplemente son armas aisladas que no comparten suficientes materiales con otras.

Estos subgrafos sirven para calcular PageRank dentro de cada comunidad y elegir sus armas representantes.


In [36]:
df_representantes = pd.DataFrame(representantes)

display(Markdown("## Paso 3. Armas representantes por comunidad"))
display(df_representantes)

## Paso 3. Armas representantes por comunidad

,comunidad,armas_representantes,pagerank,num_nodos,num_relaciones
0,0,"[Arco de cazador I, Aspa ósea I, Cuerno óseo I...",0.961240,14,182
1,1,"[Daga Barina II, Hacha Barina II]",0.988821,34,590
2,2,"[Arco de esperanza III, Cañón de esperanza III...",1.087526,35,994
3,3,"[Acelerador de hierro II, Asalto de hierro II,...",1.005936,21,252
4,4,"[Cuerno de esperanza V, Filo de esperanza V, G...",1.153031,42,1318
5,5,"[Alabarda de zorro I, Amanecer I, Campana poét...",0.961240,14,182
6,6,[Lanza de esperanza I],0.150000,1,0
7,7,"[Arco de esperanza II, Cañón de esperanza II, ...",0.961240,14,182
8,8,"[Daga Barina III, Escudo Barina III, Espada ll...",1.023586,11,110
9,9,"[Azotarraudos Dosha I, Azotederribo Dosha I, C...",0.961240,8,56


#### Resultados obtenidos

El resultado final muestra, para cada comunidad, las armas representantes, su valor de PageRank, el número de nodos y el número de relaciones internas.

Se observan comunidades grandes, como la comunidad `35`, con `128` armas y `16032` relaciones, o la comunidad `50`, con `100` armas y `9900` relaciones. Estas comunidades representan grupos amplios de armas que comparten muchos materiales entre sí.

También aparecen comunidades de un solo nodo, con `num_nodos = 1` y `num_relaciones = 0`. Esto no es un error: simplemente significa que esa arma quedó aislada como su propia comunidad. En esos casos, el PageRank toma el valor base `0.15`, y esa única arma es automáticamente la representante.

En algunas comunidades aparecen varias armas representantes. Esto ocurre cuando varias armas tienen el mismo valor máximo de PageRank, normalmente porque tienen una posición muy parecida dentro del subgrafo o comparten patrones de similitud muy similares.

In [37]:
gds.graph.drop(G_pr)
print("Grafo de PageRank eliminado de memoria")

Grafo de PageRank eliminado de memoria
